Análisis exploratorio de datos para los datos de Kaggle, uso problematico de internet

Carga de bibliotecas

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import duckdb

# Configuración estética general de las gráficas
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13

# Evita que pandas trunque columnas al visualizar
pd.set_option("display.max_columns", None)

Carga de archivos de analisis preliminar, NO REALIZAR a menos que se cuenten con los 6 gigas de datos

In [7]:
#leer el CSV
df_csv = pd.read_csv('train.csv')
df_csv['sii'].value_counts(dropna=False)


sii
0.0    1594
NaN    1224
1.0     730
2.0     378
3.0      34
Name: count, dtype: int64

Inicialmente solo se obtienen los datos tabulares de cuestionarios, sin embargo es importante integrar la informacion del acelerometro de muñeca, para ello se crearan nuevas columnas con los siguientes criterios a partir de las variables:
- enmo: representa movimiento total (en los ejes X,Y,Z).
- time_of_day: el registro del momento del dia de la informacion, es importante para separar actividades diurnas de trasnocho.
- anglez: el angulo en que se encuentra el dispositivo de la muñeca, sirve para detectar sueño vs actividad con celular en las noches.
- light: detecta luz, en el dia queremos actividad, en la noche nos sirve para detectar luz de pantallas.
- weekday: el dia se la semana de la medicion, queres discriminar dias escolares de fines de semana.
- quarter: corresponde al trimetre de las mediciones.

Usando estos datos generamos las siguientes columnas:
- enmo_general_semana: promedio de movimiento general todo el dia entre semana.
- enmo_general_fin_de_semana: promedio de movimiento general todo el dia en sabado y domingo.
- enmo_madrugada: promedio de movimiento en horas de la noche (12am a 5am), nuestro objetivo es detectar trasnocho.
- enmo_libre: promedio de actividad en horas libres (3pm que finaliza el horario escolar a 10pm)
- anglez_madrugada: cambios de la medicion en las horas que se esperan de sueño (esperamos poca actividad, muchos cambios de angulo sugieren uso de celular)
- light_dia: promedio de luz recibido en horas del dia (7am a 6pm)
- light_madrugada: promedio de luz entre 12am y 5am, sugiere exposicion a luz de pantalla en la madrugada.

Consideracion de estaciones: debido a que es poco probable que un participante use el dispositivo un año completo, no consideraremos inicialmente las estaciones. En caso que el analisis lo requiera mas adelante se incluira.

In [8]:
import duckdb

df_eda = duckdb.sql("""
    WITH datos_limpios AS (
        SELECT 
            id,
            enmo,
            anglez,
            light,
            weekday,
            -- Convierte nanosegundos a hora decimal (0.0 a 24.0)
            time_of_day / 3600000000000.0 AS hora
        FROM read_parquet('series_train.parquet/*/*.parquet', hive_partitioning=true)
        WHERE "non-wear_flag" = 0  -- Filtra solo cuando el reloj estaba puesto
    ),
    resumen_reloj AS (
        SELECT 
            id,
            -- 1. Promedio de movimiento general entre semana (lunes=1 a viernes=5)
            AVG(CASE WHEN weekday BETWEEN 1 AND 5 THEN enmo END) AS enmo_general_semana,
            
            -- 2. Promedio de movimiento general fin de semana (sábado=6 y domingo=7)
            AVG(CASE WHEN weekday IN (6, 7) THEN enmo END) AS enmo_general_fin_de_semana,
            
            -- 3. Promedio de movimiento en madrugada (12am a 5am)
            AVG(CASE WHEN hora >= 0 AND hora < 5 THEN enmo END) AS enmo_madrugada,
            
            -- 4. Promedio de actividad en horas libres (3pm = 15.0 a 10pm = 22.0)
            AVG(CASE WHEN hora >= 15 AND hora < 22 THEN enmo END) AS enmo_libre,
            
            -- 5. Cambios/variación en el ángulo del brazo en madrugada (12am a 5am)
            STDDEV(CASE WHEN hora >= 0 AND hora < 5 THEN anglez END) AS anglez_madrugada,
            
            -- 6. Promedio de luz recibida en horas del día (7am = 7.0 a 6pm = 18.0)
            AVG(CASE WHEN hora >= 7 AND hora < 18 THEN light END) AS light_dia,
            
            -- 7. Promedio de luz en madrugada (12am a 5am)
            AVG(CASE WHEN hora >= 0 AND hora < 5 THEN light END) AS light_madrugada
            
        FROM datos_limpios
        GROUP BY id
    )
    SELECT c.*, r.* EXCLUDE (id)
    FROM 'train.csv' c
    LEFT JOIN resumen_reloj r
    ON c.id = r.id
""").df()

print(f"¡Tabla integrada con éxito! Dimensiones: {df_eda.shape}")

# Muestra los primeros 10 registros que sí tienen datos de reloj para verificar
nuevas_cols = [
    'id', 'sii', 'enmo_general_semana', 'enmo_general_fin_de_semana',
    'enmo_madrugada', 'enmo_libre', 'anglez_madrugada', 'light_dia', 'light_madrugada'
]
display(df_eda[nuevas_cols].dropna(subset=['enmo_general_semana']).head(10))


¡Tabla integrada con éxito! Dimensiones: (3960, 89)


,id,sii,enmo_general_semana,enmo_general_fin_de_semana,enmo_madrugada,enmo_libre,anglez_madrugada,light_dia,light_madrugada
0,43720c15,0,0.040137,0.037720,0.044046,0.037887,37.177107,85.558809,11.557001
1,93c93585,0,0.050199,0.055259,0.016039,0.069580,42.395305,7.770470,4.994114
2,d445fb74,2,0.030149,0.028236,0.007170,0.037142,39.912505,30.738253,20.421779
3,04cb2c30,1,0.035953,0.032659,0.005775,0.067645,41.147454,30.772023,12.047099
4,2c406043,0,0.059608,0.072528,0.013897,0.073219,53.466478,239.499621,7.473101
5,92293b06,0,0.026135,0.023666,0.002895,0.032435,39.510447,29.291462,14.353986
6,d6776176,0,0.052279,0.052990,0.008564,0.080765,44.701337,17.711166,12.043590
7,012cadd8,0,0.063869,0.048173,0.004327,0.111081,46.478312,111.639043,9.737230
8,0bfe4fc9,1,0.037329,0.035376,0.007760,0.072002,44.350558,28.443336,8.073497
9,25849c6c,1,0.022314,0.019529,0.004365,0.031820,39.060027,72.372179,12.406944


A continuacion grabaremos en memoria permanente este archivo con toda la informacion que requerimos para no tener que volver a realizar todas las integraciones.

In [10]:
df_eda.to_csv('df_eda_completo.csv', index=False)